# Experimento 11 — Generalização descritiva: Random Forest × Extra Trees × Soft Voting

**Objetivo:** comparar os três classificadores em dois conjuntos externos *já inspecionados* (29/10/2020 e 15/01/2026), sem selecionar um vencedor com base neles.

**Protocolo congelado para esta análise exploratória:**
- Mesmas quatro features originais, mesmo treino (123 janelas: 23 positivas, 100 negativas).
- Reamostragem positiva **somente no treinamento**, semente 42.
- Random Forest: `n_estimators=100`, `max_depth=5`, `class_weight='balanced'`.
- Extra Trees: `n_estimators=100`, `max_depth=5`, `class_weight='balanced'`.
- Ensemble: média **0,5 × RF + 0,5 × ET** (soft voting simples), sem treinamento adicional.
- Limiar nominal `0,5` para comparações descritivas; limiar `0,85` aparece **apenas como sensibilidade adicional**, pois foi escolhido anteriormente para RF com dados de 2020.
- Não afirmar precisão/F1 reais nos conjuntos externos: endereços fora dos pools de referência **não têm rótulo negativo verificado**.
- Métricas externas principais: recuperação dos pools de referência, número de outros candidatos, posições/ranking dos pools e score de cada endereço.

**IMPORTANTE:** 2020 e 2026 já foram examinados na pesquisa. Não são mais um teste cego para selecionar modelos. O ensaio serve para diagnóstico, não confirmação de generalização. O próximo experimento confirmatório precisa de novos períodos e outros contratos reservados.


## 1 · Ambiente

In [1]:
import sys, subprocess, importlib.util, hashlib, json, time
from pathlib import Path
if importlib.util.find_spec('pyarrow') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'])
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Fora do Colab — ajuste BASE_MANUAL.')
import pandas as pd, numpy as np, sklearn, joblib
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.utils import resample
from sklearn.metrics import average_precision_score
print('scikit-learn:', sklearn.__version__)

Mounted at /content/drive
scikit-learn: 1.6.1


## 2 · Localizar a cópia correta do Drive
Se houver várias pastas com o mesmo nome, a escolha deverá ser explícita. O notebook não modifica os dados originais.

In [2]:
BASE_MANUAL = None  # exemplo: '/content/drive/MyDrive/mixers-ethereum (2)'
ESCOLHA_INDICE = None
root = Path('/content/drive/MyDrive')
candidates = []
if root.is_dir():
    candidates = [
        p for p in root.iterdir()
        if p.is_dir() and p.name.lower().startswith('mixers-ethereum')
        and (p/'data/processed/features_treino.parquet').is_file()
    ]
if BASE_MANUAL:
    BASE = Path(BASE_MANUAL)
elif len(candidates)==1:
    BASE = candidates[0]
elif len(candidates)>1:
    print('Cópias encontradas:')
    for i,p in enumerate(candidates): print(i, p)
    if ESCOLHA_INDICE is None:
        raise RuntimeError('Defina ESCOLHA_INDICE após verificar qual pasta é a original.')
    BASE = candidates[ESCOLHA_INDICE]
else:
    raise FileNotFoundError('Pasta de dados não localizada. Adicione atalho ao Meu Drive ou configure BASE_MANUAL.')
PROC = BASE/'data/processed'
TRAIN = PROC/'features_treino.parquet'
EXTERNOS = {
    '2020-10-29': PROC/'features_teste_20201029.parquet',
    '2026-01-15': PROC/'features_2026-01-15.parquet'
}
for f in [TRAIN,*EXTERNOS.values()]:
    if not f.is_file(): raise FileNotFoundError(f'Não encontrado: {f}')
print('Projeto:',BASE)
print('Arquivos confirmados:',*[str(f) for f in [TRAIN,*EXTERNOS.values()]],sep='\\n- ')


Projeto: /content/drive/MyDrive/mixers-ethereum
Arquivos confirmados:\n- /content/drive/MyDrive/mixers-ethereum/data/processed/features_treino.parquet\n- /content/drive/MyDrive/mixers-ethereum/data/processed/features_teste_20201029.parquet\n- /content/drive/MyDrive/mixers-ethereum/data/processed/features_2026-01-15.parquet


## 3 · Integridade, rótulos de referência e saída isolada

In [3]:
FEATURES=['value_entropy','fee_entropy','internal_value_entropy','call_count_mean']
TORNADO={
'0x12d66f87a04a9e220743712ce6d9bb1b5616b8fc':'Tornado 0,1 ETH',
'0x47ce0c6ed5b0ce3d3a51fdb1c52dc66a7c3c2936':'Tornado 1 ETH',
'0x910cbd523d972eb0a6f4cae4618ad62622b39dbf':'Tornado 10 ETH',
}
# quarto pool para classificação contextual, nunca contabilizado como verdade de referência dos três
T100='0xa160cdab225685da1d56aa342ad8841c3b53f291'
def sha256(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(1<<20),b''): h.update(b)
    return h.hexdigest()
train=pd.read_parquet(TRAIN)
if not set(FEATURES+['label']).issubset(train.columns):
    raise ValueError('Treino não tem quatro features e label')
for col in FEATURES:
    train[col]=pd.to_numeric(train[col],errors='raise')
if train[FEATURES].isna().any().any() or not np.isfinite(train[FEATURES].to_numpy(float)).all():
    raise ValueError('Treino contém valores ausentes ou infinitos')
if set(train.label.astype(int).unique())!={0,1}: raise ValueError('Treino precisa de classes 0 e 1')
print('Treino:',train.shape,train.label.value_counts().to_dict())
print('Treino SHA256:',sha256(TRAIN))
OUT=BASE/'results'/'11_generalizacao_diagnostica'/(
    time.strftime('%Y%m%d_%H%M%S')+'_'+sha256(TRAIN)[:8])
OUT.mkdir(parents=True,exist_ok=False)
manifest={'base':str(BASE),'train_sha256':sha256(TRAIN),
          'externos_sha256':{k:sha256(v) for k,v in EXTERNOS.items()},
          'sklearn':sklearn.__version__,
          'protocolo':'Avaliação pós-hoc descritiva; 2020 e 2026 já vistos',
          'modelos':['Random Forest','Extra Trees','Soft Voting 50/50'],
          'limiar_principal':0.5,'limiar_sensibilidade':0.85}
(OUT/'manifesto.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('Saídas:',OUT)


Treino: (123, 8) {0: 100, 1: 23}
Treino SHA256: 0a8baa11cee97d338d7a8d92b984f38192ba1a10ec915cc4a06d67ec94ad5c32
Saídas: /content/drive/MyDrive/mixers-ethereum/results/11_generalizacao_diagnostica/20261115_005204_0a8baa11


## 4 · Treino padronizado dos dois modelos (sem usar dados externos)

In [4]:
X=train[FEATURES].reset_index(drop=True)
y=train['label'].astype(int).reset_index(drop=True)
d=X.copy(); d['_label']=y
a=d[d['_label']==0]; b=d[d['_label']==1]
if len(a)>len(b):
    b=resample(b,replace=True,n_samples=len(a),random_state=42)
elif len(b)>len(a):
    a=resample(a,replace=True,n_samples=len(b),random_state=42)
bal=pd.concat([a,b],ignore_index=True)
Xb=bal[FEATURES]; yb=bal['_label']
rf=RandomForestClassifier(n_estimators=100,max_depth=5,class_weight='balanced',
                           random_state=42,n_jobs=-1).fit(Xb,yb)
et=ExtraTreesClassifier(n_estimators=100,max_depth=5,class_weight='balanced',
                         random_state=42,n_jobs=-1).fit(Xb,yb)
modelos={'Random Forest':rf,'Extra Trees':et}
print('Concluído. Classes após reamostragem:',yb.value_counts().to_dict())


Concluído. Classes após reamostragem: {0: 100, 1: 100}


## 5 · Avaliação externa descritiva
A recuperação mede apenas **três pools positivos conhecidos**. Os demais endereços são **candidatos adicionais de classe desconhecida**; não serão chamados falsos positivos. Para 2026, o dataset pode conter a coluna `rotulo='router'`, que será excluída como na análise anterior.

In [5]:
linhas=[]
predicoes=[]
for periodo,path in EXTERNOS.items():
    dados=pd.read_parquet(path).copy()
    if not set(FEATURES+['recipient','n_tx']).issubset(dados.columns):
        raise ValueError(f'Colunas incompatíveis: {path}')
    dados['recipient']=dados['recipient'].astype(str).str.lower()
    if periodo=='2026-01-15' and 'rotulo' in dados.columns:
        dados=dados[dados['rotulo'].fillna('').astype(str).str.lower()!='router']
    dados=dados[dados['n_tx']>=10].copy().reset_index(drop=True)
    for col in FEATURES:
        dados[col]=pd.to_numeric(dados[col],errors='coerce')
    dados=dados[np.isfinite(dados[FEATURES].to_numpy(float)).all(axis=1)].copy()
    if dados['recipient'].duplicated().any():
        raise ValueError(f'{periodo}: endereços repetidos; revisar granularidade janela/dia')
    dados['conhecido']=dados['recipient'].map(TORNADO)
    dados.loc[dados.recipient==T100,'outro_pool_conhecido']='Tornado 100 ETH'
    dados['periodo']=periodo
    for nome,modelo in modelos.items():
        dados[nome]=modelo.predict_proba(dados[FEATURES])[:,1]
    dados['Soft Voting 50/50']=0.5*(dados['Random Forest']+dados['Extra Trees'])
    for nome in [*modelos.keys(),'Soft Voting 50/50']:
        for limiar in [0.5,0.85]:
            sel=dados[nome]>limiar
            pools=dados['conhecido'].notna()
            recuperados=int((sel&pools).sum())
            linhas.append({'periodo':periodo,'modelo':nome,'limiar':limiar,
                  'avaliados':int(len(dados)),'pools_referencia_avaliaveis':int(pools.sum()),
                  'pools_referencia_sinalizados':recuperados,
                  'outros_candidatos_sinalizados':int((sel&~pools).sum()),
                  'total_sinalizados':int(sel.sum())})
        for rank,(_,r) in enumerate(dados.sort_values(nome,ascending=False).iterrows(),1):
            if pd.notna(r['conhecido']):
                predicoes.append({'periodo':periodo,'modelo':nome,'pool':r['conhecido'],
                                  'rank':rank,'pontuacao':r[nome]})
    cols=['periodo','recipient','n_tx','conhecido',*FEATURES,*modelos.keys(),'Soft Voting 50/50']
    dados[cols].to_csv(OUT/f'pontuacoes_{periodo}.csv',index=False)
    print(periodo,'avaliados:',len(dados),'pools elegíveis:',dados['conhecido'].notna().sum())
resumo=pd.DataFrame(linhas)
ranks=pd.DataFrame(predicoes)
resumo.to_csv(OUT/'comparacao_externa.csv',index=False)
ranks.to_csv(OUT/'ranking_pools.csv',index=False)
display(resumo)
display(ranks)


2020-10-29 avaliados: 4038 pools elegíveis: 3
2026-01-15 avaliados: 20214 pools elegíveis: 3


,periodo,modelo,limiar,avaliados,pools_referencia_avaliaveis,pools_referencia_sinalizados,outros_candidatos_sinalizados,total_sinalizados
0,2020-10-29,Random Forest,0.50,4038,3,3,6,9
1,2020-10-29,Random Forest,0.85,4038,3,3,4,7
2,2020-10-29,Extra Trees,0.50,4038,3,3,23,26
3,2020-10-29,Extra Trees,0.85,4038,3,0,2,2
4,2020-10-29,Soft Voting 50/50,0.50,4038,3,3,11,14
5,2020-10-29,Soft Voting 50/50,0.85,4038,3,0,2,2
6,2026-01-15,Random Forest,0.50,20214,3,3,12,15
7,2026-01-15,Random Forest,0.85,20214,3,3,6,9
8,2026-01-15,Extra Trees,0.50,20214,3,3,54,57
9,2026-01-15,Extra Trees,0.85,20214,3,3,4,7


,periodo,modelo,pool,rank,pontuacao
0,2020-10-29,Random Forest,Tornado 1 ETH,4,0.870000
1,2020-10-29,Random Forest,"Tornado 0,1 ETH",5,0.870000
2,2020-10-29,Random Forest,Tornado 10 ETH,7,0.860000
3,2020-10-29,Extra Trees,"Tornado 0,1 ETH",5,0.754857
4,2020-10-29,Extra Trees,Tornado 10 ETH,6,0.752485
5,2020-10-29,Extra Trees,Tornado 1 ETH,8,0.744857
6,2020-10-29,Soft Voting 50/50,"Tornado 0,1 ETH",5,0.812429
7,2020-10-29,Soft Voting 50/50,Tornado 1 ETH,6,0.807429
8,2020-10-29,Soft Voting 50/50,Tornado 10 ETH,7,0.806243
9,2026-01-15,Random Forest,Tornado 1 ETH,1,1.000000


## 6 · Divergências entre classificadores, sem criar rótulos

In [6]:
for periodo in EXTERNOS:
    df=pd.read_csv(OUT/f'pontuacoes_{periodo}.csv')
    for limiar in [0.5,0.85]:
        a=df['Random Forest']>limiar
        b=df['Extra Trees']>limiar
        cols=['recipient','n_tx','conhecido','Random Forest','Extra Trees','Soft Voting 50/50']
        diverg=df.loc[a!=b,cols].copy()
        diverg['periodo']=periodo
        diverg['limiar']=limiar
        diverg.to_csv(OUT/f'divergencias_{periodo}_{str(limiar).replace(".","_")}.csv',index=False)
        print(periodo,'limiar',limiar,'discordâncias:',len(diverg))


2020-10-29 limiar 0.5 discordâncias: 17
2020-10-29 limiar 0.85 discordâncias: 5
2026-01-15 limiar 0.5 discordâncias: 42
2026-01-15 limiar 0.85 discordâncias: 2


## 7 · Relatório automático e ZIP
O relatório não irá escolher vencedor com base nos dois dias explorados.

In [7]:
from zipfile import ZipFile,ZIP_DEFLATED
txt=[
'# Experimento 11 — Comparação diagnóstica entre modelos',
'',
'**Natureza:** exploratória/pós-hoc; dias externos já conhecidos no projeto.',
'',
'**Treinamento:** 123 amostras, 23 positivas de um único contrato, 100 negativas; reamostragem dentro do treino.',
'',
'**Protocolo:** Random Forest e Extra Trees (100 árvores, profundidade 5); Soft Voting igualitário sem tuning.',
'',
'## Contagem de sinalizados',
'',
resumo.to_markdown(index=False),
'',
'## Ranking dos pools de referência',
'',
ranks.to_markdown(index=False),
'',
'## Limitações',
'',
'- Os seis (ou outros) candidatos sem rótulo verificado NÃO são falsos positivos confirmados.',
'- 2020 já foi usado para seleção do limiar e 2026 já foi inspecionado; não usar esta comparação como prova confirmatória.',
'- Os positivos de treino vêm de um só contrato, portanto nenhuma métrica prova generalização para mixers desconhecidos.',
'- Não selecionar modelo vencedor ou limiar no teste e depois apresentar esses mesmos testes como independentes.',
'- Próximo passo: obter rótulos verificados de mais pools e novos períodos reservados.',
]
(OUT/'relatorio.md').write_text('\\n'.join(txt),encoding='utf-8')
zip_path=OUT.parent/(OUT.name+'.zip')
with ZipFile(zip_path,'w',compression=ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file(): z.write(p,arcname=p.relative_to(OUT))
print('Relatório:',OUT/'relatorio.md')
print('ZIP:',zip_path)


Relatório: /content/drive/MyDrive/mixers-ethereum/results/11_generalizacao_diagnostica/20261115_005204_0a8baa11/relatorio.md
ZIP: /content/drive/MyDrive/mixers-ethereum/results/11_generalizacao_diagnostica/20261115_005204_0a8baa11.zip
